# Day 32 — Error Handling + Logging in Data Pipelines
**Estimated time:** 90 minutes  
**Learning objectives:**
1. Apply structured try/except patterns specific to data work (CSV parsing, type errors, DB failures)
2. Configure the Python logging module for pipeline observability
3. Build a pipeline that continues on row-level errors while logging all failures

---
> **Why this matters at enterprise analytics:** On client engagements, data pipelines run against messy source extracts — SAP flat files with encoding issues, missing columns, date format inconsistencies. A pipeline that crashes at 3am with no log context is a production incident. One that logs every anomaly and continues is a maintainable professional deliverable.


## 1. Logging Configuration — The Right Way

In [ ]:
import logging
import sys
from pathlib import Path

def setup_logging(
    name: str = 'pipeline',
    level: int = logging.INFO,
    log_file: Path | None = None
) -> logging.Logger:
    """Configure and return a named logger with console and optional file output.

    Args:
        name: Logger name (use __name__ in production modules).
        level: Log level (logging.DEBUG, INFO, WARNING, ERROR, CRITICAL).
        log_file: Optional path to write logs to file as well as console.

    Returns:
        Configured Logger instance.
    """
    logger = logging.getLogger(name)
    logger.setLevel(level)

    # Avoid duplicate handlers if called multiple times
    if logger.handlers:
        logger.handlers.clear()

    fmt = logging.Formatter(
        '%(asctime)s | %(levelname)-8s | %(name)s | %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S'
    )

    # Console handler
    ch = logging.StreamHandler(sys.stdout)
    ch.setLevel(level)
    ch.setFormatter(fmt)
    logger.addHandler(ch)

    # File handler (optional)
    if log_file:
        fh = logging.FileHandler(log_file)
        fh.setLevel(logging.DEBUG)  # log everything to file
        fh.setFormatter(fmt)
        logger.addHandler(fh)

    return logger


log = setup_logging('pipeline', logging.DEBUG, log_file=Path('pipeline.log'))
log.info('Logger initialised.')
log.debug('Debug mode active — verbose output enabled.')
log.warning('This is what a WARNING looks like.')

## 2. Try/Except Patterns for Data Work

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Pattern 1: Safe file loading with meaningful errors
def safe_load_csv(path: str | Path, **kwargs) -> pd.DataFrame | None:
    """Load a CSV file with structured error handling."""
    path = Path(path)
    try:
        df = pd.read_csv(path, **kwargs)
        log.info('Loaded %s: %d rows', path.name, len(df))
        return df
    except FileNotFoundError:
        log.error('File not found: %s', path)
        return None
    except pd.errors.EmptyDataError:
        log.error('File is empty: %s', path)
        return None
    except pd.errors.ParserError as e:
        log.error('CSV parse error in %s: %s', path.name, e)
        return None
    except UnicodeDecodeError as e:
        log.warning('Encoding error in %s — retrying with latin-1: %s', path.name, e)
        try:
            return pd.read_csv(path, encoding='latin-1', **kwargs)
        except Exception as e2:
            log.error('Retry also failed: %s', e2)
            return None

# Pattern 2: Safe type conversion
def safe_to_numeric(series: pd.Series, col_name: str) -> pd.Series:
    """Convert a series to numeric, logging any coerced values."""
    converted = pd.to_numeric(series, errors='coerce')
    n_bad = converted.isna().sum() - series.isna().sum()
    if n_bad > 0:
        log.warning('Column %s: %d values could not be converted to numeric', col_name, n_bad)
    return converted

# Pattern 3: Safe date parsing
def safe_to_datetime(series: pd.Series, col_name: str, fmt: str | None = None) -> pd.Series:
    """Parse dates with logging on failures."""
    converted = pd.to_datetime(series, format=fmt, errors='coerce')
    n_bad = converted.isna().sum() - series.isna().sum()
    if n_bad > 0:
        log.warning('Column %s: %d values failed date parsing', col_name, n_bad)
        # Log examples of bad values
        bad_vals = series[converted.isna() & series.notna()].head(3).tolist()
        log.debug('  Bad examples: %s', bad_vals)
    return converted

# Test the patterns
from analytics_bootcamp.config import DATASETS
df = safe_load_csv(DATASETS['sales_orders'])
df['ERDAT'] = safe_to_datetime(df['ERDAT'], 'ERDAT')
df['NETWR'] = safe_to_numeric(df['NETWR'], 'NETWR')
print('Data loaded cleanly:', df.shape)

## 3. Handling Malformed Data Gracefully — Row-Level Error Logging

In [ ]:
from analytics_bootcamp.dataset import load_all
datasets = load_all()
# YOUR SOLUTION
# Build a processor that applies a transformation function row-by-row,
# continuing on errors, collecting all failures in an error log.

from typing import Callable, Any
import traceback

class RowProcessor:
    """Applies a function to each row, logging errors without stopping the pipeline."""

    def __init__(self, name: str) -> None:
        self.name = name
        self.logger = logging.getLogger(f'RowProcessor.{name}')
        self.errors: list[dict] = []

    def process(
        self,
        df: pd.DataFrame,
        fn: Callable[[pd.Series], Any],
        output_col: str
    ) -> pd.DataFrame:
        """Apply fn to each row, storing results in output_col.
        Rows that raise exceptions get None and are logged."""
        results = []
        for idx, row in df.iterrows():
            try:
                results.append(fn(row))
            except Exception as e:
                self.logger.warning('Row %s failed: %s — %s', idx, type(e).__name__, e)
                self.errors.append({'row_index': idx, 'error_type': type(e).__name__,
                                    'error_msg': str(e), 'row_data': row.to_dict()})
                results.append(None)

        df = df.copy()
        df[output_col] = results
        self.logger.info('Processed %d rows: %d OK, %d errors',
                         len(df), len(df) - len(self.errors), len(self.errors))
        return df

    @property
    def error_log(self) -> pd.DataFrame:
        return pd.DataFrame(self.errors)


# Example: calculate inventory value, but simulate some bad data
mat = datasets['materials']

# Inject some bad data for testing
mat_test = mat.copy()
mat_test.loc[5, 'STPRS'] = 'INVALID'   # bad price
mat_test.loc[12, 'LABST'] = None        # null qty
mat_test['STPRS'] = pd.to_numeric(mat_test['STPRS'], errors='coerce')  # will NaN the invalid

def calc_inv_value(row: pd.Series) -> float:
    if pd.isna(row['LABST']) or pd.isna(row['STPRS']):
        raise ValueError(f'Missing value: LABST={row["LABST"]}, STPRS={row["STPRS"]}')
    return row['LABST'] * row['STPRS']

proc = RowProcessor('inventory_value')
result_df = proc.process(mat_test.head(20), calc_inv_value, 'CALC_INV_VALUE')

print(f'Error count: {len(proc.errors)}')
if proc.errors:
    print(proc.error_log[['row_index','error_type','error_msg']])

## 4. Putting It Together — Observable Pipeline

In [ ]:
from analytics_bootcamp.config import RAW_DATA_DIR
# YOUR SOLUTION
# Build a pipeline that: loads data, validates, transforms, reports errors

def run_observable_pipeline(data_path: Path = RAW_DATA_DIR) -> dict:
    """Full pipeline with error handling and structured logging throughout."""
    log = logging.getLogger('observable_pipeline')
    summary = {'loaded': 0, 'errors': 0, 'warnings': 0, 'output_rows': 0}

    # Step 1: Load
    log.info('=== STEP 1: Load ===')
    sales = safe_load_csv(data_path / 'sales_orders.csv')
    if sales is None:
        log.critical('Cannot proceed without sales data.')
        summary['errors'] += 1
        return summary
    summary['loaded'] += len(sales)

    # Step 2: Parse dates
    log.info('=== STEP 2: Parse Dates ===')
    sales['ERDAT'] = safe_to_datetime(sales['ERDAT'], 'ERDAT')
    bad_dates = sales['ERDAT'].isna().sum()
    summary['warnings'] += bad_dates

    # Step 3: Calculate metrics
    log.info('=== STEP 3: Calculate ===')
    open_orders = sales[sales['STATUS'] == 'OPEN'].copy()
    open_orders['ORDER_AGE'] = (pd.Timestamp('2026-04-01') - open_orders['ERDAT']).dt.days
    summary['output_rows'] = len(open_orders)

    # Step 4: Summary
    log.info('=== STEP 4: Summary ===')
    log.info('Open orders: %d | Total backlog value: $%,.0f',
             len(open_orders), open_orders['NETWR'].sum())

    return summary

result = run_observable_pipeline()
print('\nPipeline result:', result)

## Daily Prompt
**Answer independently:**

1. The `RowProcessor.process()` method uses `iterrows()`, which is slow. Rewrite it to use `apply()` while still capturing per-row errors and storing them in `self.errors`.
2. Add a `@retry(max_attempts=3, delay_seconds=1)` decorator to `safe_load_csv` that retries on `IOError`. Use `functools.wraps` to preserve the function signature.
3. What is the difference between `logging.exception()` and `logging.error()` when called inside an except block? When would you use each one?
